## Text input

https://platform.openai.com/docs/models

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from langchain.agents import create_agent

agent = create_agent(
    model='ollama:llava',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [17]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

 As a science fiction writer, I can create a capital city for the Moon based on the user's request. The Moon is a natural satellite of Earth, and it has been explored and inhabited by humans for many years. The capital city of the Moon could be a bustling metropolis, with towering skyscrapers, advanced technology, and a unique atmosphere.

The city could be named Lunopolis and be located in the Marius Hills, one of the largest and most prominent impact craters on the Moon. Lunopolis would be a hub for scientific research, lunar mining, and space exploration. It would be a city that is built to withstand the harsh lunar environment and would be powered by a combination of nuclear reactors and solar panels.

The architecture of Lunopolis would be a blend of modern and futuristic styles, with buildings constructed from lunar regolith and other Moon-derived materials. The city would be connected by a network of lunar trains and rovers, allowing for easy transportation throughout the Moon.


## Image input

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.jpg', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [19]:
print(uploader.value)

({'name': 'lunar-10056714_1280.jpg', 'type': 'image/jpeg', 'size': 433052, 'content': <memory at 0x000001F6A8B431C0>, 'last_modified': datetime.datetime(2026, 8, 28, 20, 57, 38, 157000, tzinfo=datetime.timezone.utc)},)


In [20]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [21]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/jpg"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

 The image you've provided appears to be an artistic illustration of a futuristic capital city. Here's a fictional description based on the elements present in the image:

The city is called "Cythera Prime," a name that suggests a first-of-its-kind or primary city in a future setting. It's a sprawling metropolis with a dense urban environment, featuring modern architecture with a distinctive retro-futuristic aesthetic. The architecture has a blend of natural elements, such as the palm trees, and artificial structures, including the domed buildings that could be interpreted as greenhouses or part of an advanced infrastructure.

The city is surrounded by lush greenery, which adds a sense of vibrancy and suggests a focus on sustainable urban planning. There's a clear sky overhead, indicating good weather conditions.

Cythera Prime is located at the edge of a vast body of water, which could be an ocean or sea. This strategic location suggests the city is well-connected to other parts of th

## Audio input

In [29]:
import sounddevice as sd
from scipy.io.wavfile import write
# import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
# wav_bytes = buf.getvalue()

# aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")
buf.seek(0)

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.88it/s]

Done.


0

In [30]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from faster_whisper import WhisperModel

# 1. Transcribe directly from the in-memory buffer (RAM)
print("Transcribing audio from memory...")
stt_model = WhisperModel("base", device="cpu", compute_type="int8")
segments, _ = stt_model.transcribe(buf)
transcript_text = " ".join([segment.text for segment in segments]).strip()
print(f"Transcript: {transcript_text}")

# --- COMMENTED OUT FROM TUTORIAL (Cloud Model) ---
# agent = create_agent(
#     model='gpt-audio',  # Or 'gpt-4o-audio-preview'
# )

# 2. Local Ollama Agent
agent = create_agent(
    model='ollama:llama3.1:8b',
    system_prompt="You are a helpful assistant.",
)

# --- COMMENTED OUT FROM TUTORIAL (Audio Payload) ---
# multimodal_question = HumanMessage(content=[
#     {"type": "text", "text": "Tell me about this audio file"},
#     {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
# ])

# 3. Formatted text payload containing the transcription
multimodal_question = HumanMessage(content=[
    {
        "type": "text", 
        "text": f"The user recorded an audio message that transcribed to: '{transcript_text}'. Please respond to it."
    }
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Transcribing audio from memory...
Transcript: Write a poem for me about a cat.
Here's a little poem about a cat:

Whiskers twitch, eyes so bright,
Our feline friend, a wondrous sight.
With fur as soft as silk to touch,
And purrs that soothe the heart so much.

She pads through shadows, silent as can be,
Her little nose, a twitching spree.
With claws that grasp, and a playful leap,
Our kitty queen, in slumber deep.

Her eyes, like jewels, shining bright and green,
Reflecting wisdom, in a gentle sheen.
Her soft, sweet voice, a melodic sound,
As she purrs and cuddles, without a bound.

So here's to our feline friend, so dear and true,
A loyal companion, through and through.
May her soft purrs and cuddles bring us cheer,
And her loving presence, banish all fear.
